## test sito web tse umap per vedere pattern nascosti tra le features

In [8]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
import json

# ============================================================
# LOAD SUBJECT TENSOR
# ============================================================

project_root = Path("/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech")

SUBJECT_ID = 55
pt_path = project_root / "data/processed/subject_tensors/subject_tensors_time" / f"subject_{SUBJECT_ID:02d}.pt"

obj = torch.load(pt_path, map_location="cpu")

X = obj["X"].numpy()              # (n_trials, n_windows, n_channels, n_features)
y = obj["y"].numpy()
subject_ids = obj["subject_id"].numpy()
session_ids = obj["session_id"].numpy()
epoch_ids = obj["epoch_idx"].numpy()

print("X shape:", X.shape)

# ============================================================
# LABEL MAP (110 parole)
# ============================================================

label2idx_path = project_root / "data/interim/label2idx.json"
with open(label2idx_path, "r") as f:
    label2idx = json.load(f)

idx2label = {int(v): k for k, v in label2idx.items()}
word_labels = [idx2label.get(int(lbl), f"UNK_{lbl}") for lbl in y]

# ============================================================
# FLATTEN ONLY
# ============================================================

n_samples = X.shape[0]

# (n_samples, 5, 59, 40) -> (n_samples, 11800)
X_flat = X.reshape(n_samples, -1)

print("X_flat shape:", X_flat.shape)

# ============================================================
# SAVE FOR TENSORFLOW PROJECTOR
# ============================================================

out_dir = project_root / "notebooks" / "tests" / "projector" / f"subject_{SUBJECT_ID:02d}_raw"
out_dir.mkdir(parents=True, exist_ok=True)

vectors_path = out_dir / "vectors.tsv"
metadata_path = out_dir / "metadata.tsv"

np.savetxt(vectors_path, X_flat, delimiter="\t", fmt="%.6f")

metadata = pd.DataFrame({
    "sample_id": np.arange(n_samples),
    "word": word_labels,
    "label_id": y,
    "subject_id": subject_ids,
    "session_id": session_ids,
    "epoch_idx": epoch_ids,
})

metadata.to_csv(metadata_path, sep="\t", index=False)

print("Saved:")
print(vectors_path)
print(metadata_path)

X shape: (550, 5, 59, 40)
X_flat shape: (550, 11800)
Saved:
/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/notebooks/tests/projector/subject_55_raw/vectors.tsv
/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/notebooks/tests/projector/subject_55_raw/metadata.tsv


Cosa ottengo? 
1. la struttura della manifold NON è guidata dalla sessione.

Quindi:

✅ niente forte session drift
✅ niente cluster per sessione
✅ le feature sono abbastanza stabili nel tempo

2. l’ordine temporale del trial non guida la struttura.

Quindi:
	•	niente fatigue effect evidente
	•	niente drift progressivo
	•	niente learning effect del soggetto

3. Quindi cosa sta spiegando la manifold?

Non:
	•	parola
	•	sessione
	•	tempo
Quindi cosa? lo stato cerebrale del trial.

4. La forma a anello

Questo è interessante.

Quando UMAP produce una struttura tipo: ring / loop / horseshoe
spesso significa che i dati stanno su una manifold continua.

Esempi classici:
	•	ciclo sonno-veglia
	•	oscillazioni alpha
	•	stato attentivo
	•	dinamica temporale

Non sono cluster discreti.

Sono stati continui.